# 10 · RL with a KL penalty and an adjusted reward (`grpo_v2`, `grpo_v3`)

**Why a new RL run.** Our first RL model (`grpo_v1`, notebook 06) raised the share of legal answers to 82%, the highest of any model, but puzzle accuracy fell from 7.8% (SFT) to 3.3%. It learned to play quiet pawn moves (41% of its legal answers, vs. 2% for SFT), which are almost always legal but rarely solve a puzzle. Two features of its setup invite this:
1. **Nothing kept it close to the SFT model.** With no KL penalty (β = 0), RL was free to drift as far as the reward pushed it.
2. **The reward made legality worth more than move quality.** An illegal answer scored −1 and every legal answer at least +0.1, while the best and the worst legal move differ by only 1. A safe legal move was the easiest way to raise the reward.

**The KL penalty in one paragraph.** The *KL divergence* measures how far the RL model's probabilities over answers have moved from those of the SFT model. GRPO subtracts β × KL from the objective, so moving away from SFT has a cost; β sets how large. With β > 0, the model can still change, but only where the reward gain is worth the distance. The SFT model is the reference (TRL keeps a frozen copy of the starting adapter for this).

**What changes.** Two runs, each starting from `sft_v1` and trained on the same 2,000 positions as `grpo_v1`:

| | `grpo_v1` (notebook 06) | `grpo_v2` | `grpo_v3` |
|---|---|---|---|
| KL penalty β | 0 (none) | **0.04** | **0.04** |
| Reward: illegal or unparseable answer | −1 | −1 | **−0.5** |
| Reward: legal move | 0.1 + (1 − loss/1000) | the same | **1 − loss/1000** (no bonus just for being legal) |
| Everything else (positions, 8 answers each, learning rate, LoRA) | `configs/rl.yaml` | the same | the same |

So `grpo_v2` vs. `grpo_v1` measures the effect of the KL penalty, and `grpo_v3` vs. `grpo_v2` the effect of the reward change. In `grpo_v3` the worst legal move (0) still beats an illegal answer (−0.5), but the gap between them is half the gap between the worst and best legal move.

**Running it on Databricks**
1. Open this notebook in the shared Git folder and pick **Serverless GPU → Small (1×A10)**. Medium and Large cost much more and are not needed here.
2. Set `QUICK_CHECK = True` in the Settings cell and click **Run all**. This takes a few minutes, uses tiny sizes, and saves nothing to `results/`. It catches setup problems before the long run.
3. Set `QUICK_CHECK = False` and run the real thing. For a run this long, use a **Job** so the browser can close: New → Job → type *Notebook* → this notebook → compute *Serverless GPU, 1×A10*; add an email notification.
4. Afterwards, commit `results/` from the Git folder; Databricks cannot commit notebook outputs. Then **File → Export → IPython Notebook**, replace this notebook with the exported file on a laptop, and commit it there, so the figures also show inside the notebook on GitHub.

**Time:** about 40 minutes of training and 5 minutes of grading per run, so about 1.5 hours in total.

**Outputs**
- the adapters in `checkpoints/grpo_v2/` and `checkpoints/grpo_v3/` (not on GitHub)
- results: `results/grpo_v2*.json`, `results/grpo_v3*.json`, and new rows in `results/summary.csv`
- figures and tables in `results/figures/grpo_v2/` and `results/figures/grpo_v3/`

In [ ]:
# On Databricks: install our packages and Stockfish for this session. Does nothing on other machines.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))  # so Databricks finds our code in src/
from chesslm.databricks import setup_databricks

setup_databricks(require_gpu=True)  # stops with a clear message if no GPU is attached

In [ ]:
# Settings
QUICK_CHECK = True    # True = a few-minute run with tiny sizes, to check that everything works. Saves nothing to results/.
RUNS = {              # run name -> its changes to configs/rl.yaml
    "grpo_v2": {"training": {"beta": 0.04}},
    "grpo_v3": {"training": {"beta": 0.04}, "reward": {"invalid": -0.5, "legal_bonus": 0.0}},
}

import copy
import gc

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display

pd.set_option("display.width", 200)  # print wide tables on one line

from chesslm import experiments, plots
from chesslm.config import REPO_ROOT, load_config
from chesslm.evaluate import load_eval_model
from chesslm.experiments import (compare_with, figures_dir, grade_puzzles, load_graded, move_kinds, puzzle_table,
                                 save_table, save_training)
from chesslm.rewards import to_rl_dataset
from chesslm.train_rl import train

sft_cfg, prompt_cfg, eval_cfg = load_config("sft"), load_config("prompt"), load_config("eval")
rl_cfg = load_config("rl")
configs = {"model": sft_cfg["model"], "prompt": prompt_cfg, "eval": eval_cfg}
START_ADAPTER = REPO_ROOT / rl_cfg["training"]["start_adapter"]   # sft_v1

if QUICK_CHECK:
    experiments.use_quick_check_folder()
    RUNS = {f"{name}_check": change for name, change in RUNS.items()}
    rl_cfg["training"].update(num_positions=8, logging_steps=1)
PUZZLE_LIMIT = 5 if QUICK_CHECK else None
print(f"Runs: {list(RUNS)} | starting from {START_ADAPTER}")

## 1. Training positions

The same 2,000 training positions as `grpo_v1` (the same seeded sample), with the usual prompt. Each position also carries its FEN, which the reward uses to score the answers.

In [ ]:
t = rl_cfg["training"]
train_positions = pd.read_parquet(REPO_ROOT / "data" / "processed" / "train.parquet")
train_ds = to_rl_dataset(train_positions.sample(n=t["num_positions"], random_state=t["seed"]), prompt_cfg)
print(f"{len(train_ds):,} positions x {t['num_generations']} answers each = "
      f"{len(train_ds) * t['num_generations']:,} answers scored by Stockfish\n")
print(train_ds[0]["prompt"][0]["content"])

## 2. Train and grade `grpo_v2` and `grpo_v3`

For each run, two figures:
- **average reward** of the sampled answers over training. It should rise. A rise alone does not mean better chess: `grpo_v1`'s reward rose while its puzzle accuracy fell.
- **KL divergence** from the SFT model over training: how far the model has moved. With β > 0 it should stay small instead of growing steadily.

Each model is then graded on the dev and test puzzles exactly like every other model (greedy decoding, even though training sampled answers).

In [ ]:
for run_name, change in RUNS.items():
    cfg = copy.deepcopy(rl_cfg)
    for section, values in change.items():
        cfg[section].update(values)
    adapter_dir = REPO_ROOT / "checkpoints" / run_name
    fig = figures_dir(run_name)
    print(f"=== {run_name}: beta {cfg['training']['beta']} | reward {cfg['reward']} ===")

    log, compute = train(train_ds, cfg, eval_cfg["stockfish"], sft_cfg["model"], sft_cfg["lora"], START_ADAPTER,
                         adapter_dir)
    save_training(run_name, {"rl": cfg, "prompt": prompt_cfg}, compute, log, start_adapter=START_ADAPTER)
    rewards = log.reward.dropna()
    print(f"Average reward: {rewards.iloc[0]:.3f} at the start -> {rewards.iloc[-1]:.3f} at the end")
    plots.reward_curve(log, fig / "reward.png", title=f"{run_name}: average reward of the sampled answers")
    if "kl" in log:
        plots.training_curve(log, "kl", "KL divergence from the SFT model", fig / "kl.png",
                             title=f"{run_name}: how far the model moves from SFT")
    plt.show()

    gc.collect()
    torch.cuda.empty_cache()
    tokenizer, model = load_eval_model(sft_cfg["model"], adapter_path=adapter_dir)
    grade_puzzles(model, tokenizer, "dev", f"{run_name}_dev", configs, str(adapter_dir), PUZZLE_LIMIT)
    grade_puzzles(model, tokenizer, "test", run_name, configs, str(adapter_dir), PUZZLE_LIMIT)
    del model
    gc.collect()
    torch.cuda.empty_cache()

## 3. Results: SFT vs. the three RL models

**How to read this section**
- **Puzzle accuracy** is the main metric (a random legal move solves 5.6%). RL starts from `sft_v1`, so `sft_v1` is the line to beat.
- **Legal answers:** `grpo_v1` reached 82%. The new runs may give up some of that in exchange for better moves.
- **What kind of moves** each model plays, among its legal test answers: the share of pawn moves, captures, and checks. Puzzles are mostly solved by captures and checks; `grpo_v1` shifted towards pawn moves.
- The **paired comparisons** count puzzles solved by only one of two models; a p-value below 0.05 means the difference is unlikely to be chance.

All numbers are on the 1,000 test puzzles.

In [ ]:
v2, v3 = RUNS
runs = ["sft_v1", "grpo_v1", v2, v3]
summary = pd.read_csv(experiments.RESULTS / "summary.csv")
comparison = pd.DataFrame([compare_with(v2, "sft_v1"), compare_with(v2, "grpo_v1"),
                           compare_with(v3, "sft_v1"), compare_with(v3, v2)]).set_index("model")

puzzles = pd.read_csv(REPO_ROOT / "data" / "eval" / "puzzles_test.csv").set_index("puzzle_id")
kinds = pd.DataFrame({run: move_kinds(g, puzzles.fen.loc[g.puzzle_id]) for run, g in
                      ((run, load_graded(run)) for run in runs)}).T.round(1)
kinds.columns = ["Pawn moves (%)", "Captures (%)", "Checks (%)"]
kinds.index.name = "Model"

for run_name in RUNS:  # the same comparison is saved in each new run's folder
    fig = figures_dir(run_name)
    table = puzzle_table(runs, fig)
    save_table(comparison, fig, "paired_comparison",
               "Puzzles solved by only one of the two models, and the sign-test p-value (below 0.05 = unlikely to be chance).")
    save_table(kinds, fig, "move_kinds", "Share of each model's legal test answers that are pawn moves, captures, or checks.")
    plt.close("all")  # only the last copy of each figure is shown below
    plots.compare_metrics(summary, runs, fig / "puzzle_metrics.png", title="SFT and the three RL models")
    plots.accuracy_by_band(summary, runs, fig / "accuracy_by_band.png", title="Puzzle accuracy by puzzle difficulty")
    plots.outcome_breakdown(summary, runs, fig / "answer_outcomes.png", title="Are the answers legal moves?")
    plots.metric_panels(kinds, [(c, c, "{:.0f}") for c in kinds.columns], fig / "move_kinds.png",
                        title="What kind of moves does each model play? (legal test answers)")

display(table)
display(comparison)
display(kinds)

## Summary of outputs

- `results/figures/grpo_v2/` and `results/figures/grpo_v3/`: `reward.png`, `kl.png`, `puzzle_results` (table), `paired_comparison` (table), `move_kinds` (table and figure), `puzzle_metrics.png`, `accuracy_by_band.png`, `answer_outcomes.png`
- `results/grpo_v2.json`, `results/grpo_v3.json`: every test answer and its grade; `*_training.json`: settings, compute, training log
- Both models are also graded on held-out game positions and in full games in notebook 11.